In [25]:
from federated_rsf.models import LocalRandomSurvivalForest, FederatedRandomSurvivalForest
from federated_rsf.schema import DatasetSchema, SchemaAligner, SchemaCreator
from federated_rsf.testing import federate_data
from sksurv.datasets import load_veterans_lung_cancer
from sksurv.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

n_clients = 5

X, Y = load_veterans_lung_cancer()
X_list, Y_list = federate_data(X, Y, clients=n_clients, random_state=0)
X_list = [OneHotEncoder().fit_transform(X) for X in X_list]

schema_list = [DatasetSchema(X.columns) for X in X_list]
schema_creator = SchemaCreator()
federated_schemas = schema_creator.fit_transform(schema_list)

X_aligned_list = []
local_schema_aligners = []
for X_local, schema in zip(X_list, federated_schemas):
    aligner = SchemaAligner().fit(schema)
    X_aligned = aligner.transform(X_local)
    X_aligned_list.append(X_aligned)
    local_schema_aligners.append(aligner)


In [33]:
local_models = [LocalRandomSurvivalForest(random_state=0, update_method='constant') for _ in range(n_clients)]

X_trains, X_tests, Y_trains, Y_tests = [], [], [], []

for X_local, Y_local, local_model in zip(X_aligned_list, Y_list, local_models):

    X_train, X_test, Y_train, Y_test = train_test_split(X_local, Y_local, test_size=0.3, random_state=0)
    X_trains.append(X_train)
    X_tests.append(X_test)
    Y_trains.append(Y_train)
    Y_tests.append(Y_test)

    local_model.fit(X_train, Y_train)



In [34]:
federated_model = FederatedRandomSurvivalForest(local_models=local_models)
federated_model.distribute_trees()

[LocalRandomSurvivalForest(update_method='constant'),
 LocalRandomSurvivalForest(update_method='constant'),
 LocalRandomSurvivalForest(update_method='constant'),
 LocalRandomSurvivalForest(update_method='constant'),
 LocalRandomSurvivalForest(update_method='constant')]

In [37]:

for i, (local_model, X_test, Y_test) in enumerate(zip(local_models, X_tests, Y_tests)):
    local_model.use_local_estimators()
    c_index = local_model.score(X_test, Y_test)
    print(f"Client {i+1}")
    print(f"Local model C-index: {c_index:.4f}")
    local_model.use_federated_estimators()
    federated_c_index = local_model.score(X_test, Y_test)
    print(f"Federated model C-index: {federated_c_index:.4f}")
    print(f"Number of estimators in federated model: {local_model.n_estimators}")
    print("-" * 30)

Client 1
Local model C-index: 0.6970
Federated model C-index: 0.7576
Number of estimators in federated model: 100
------------------------------
Client 2
Local model C-index: 0.5556
Federated model C-index: 0.6111
Number of estimators in federated model: 100
------------------------------
Client 3
Local model C-index: 0.7419
Federated model C-index: 0.6774
Number of estimators in federated model: 100
------------------------------
Client 4
Local model C-index: 0.4706
Federated model C-index: 0.5000
Number of estimators in federated model: 100
------------------------------
Client 5
Local model C-index: 0.6667
Federated model C-index: 0.7222
Number of estimators in federated model: 100
------------------------------
